In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [4]:
# ── 1. 의존성 설치 ────────────────────────────────────────────────────────────
import subprocess
subprocess.run([
    "pip", "install", "--quiet",
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
], check=True)
subprocess.run([
    "pip", "install", "--quiet", "--no-deps",
    "trl", "peft", "accelerate", "bitsandbytes"
], check=True)
print('설치 완료')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 123.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 646.8/646.8 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 112.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 100.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 97.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2025.9.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2025.9.0 which is incompatible.


설치 완료


In [5]:
# ── 2. 라이브러리 임포트 ──────────────────────────────────────────────────────
import os
import json
import random
from pathlib import Path

import numpy as np
import cv2
from PIL import Image
from datasets import Dataset
from kaggle_secrets import UserSecretsClient

from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

random.seed(42)
print('임포트 완료')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
임포트 완료


In [6]:
# ── 3. 데이터셋 경로 & 구조 확인 ─────────────────────────────────────────────
DFU_DIR = Path("/kaggle/input/datasets/laithjj/diabetic-foot-ulcer-dfu")
SEG_DIR = Path("/kaggle/input/datasets/leoscode/wound-segmentation-images")

print("=== DFU Binary Dataset ===")
for p in sorted(DFU_DIR.iterdir())[:20]:
    print(f"  {p.relative_to(DFU_DIR)}")

print("\n=== Segmentation Dataset ===")
for p in sorted(SEG_DIR.iterdir())[:20]:
    print(f"  {p.relative_to(SEG_DIR)}")

=== DFU Binary Dataset ===
  DFU

=== Segmentation Dataset ===
  data_wound_seg


In [7]:
# ── 4. 데이터 전처리 ──────────────────────────────────────────────────────────
ULCER_KEYWORDS   = {"ulcer", "dfu", "wound", "positive", "diabetic"}
HEALTHY_KEYWORDS = {"healthy", "normal", "non_dfu", "negative", "control"}

PROMPT = (
    "Analyze this diabetic foot image. "
    "Determine: (1) wound presence, "
    "(2) wound coverage percentage, "
    "(3) severity score 0–10, "
    "(4) recommended action."
)


def area_to_severity(ratio: float) -> int:
    if ratio < 0.02: return 3
    if ratio < 0.05: return 5
    if ratio < 0.10: return 7
    return 9


def make_sample(img_path: str, is_ulcer: bool, wound_ratio: float = 0.0) -> dict:
    if is_ulcer:
        sev = area_to_severity(wound_ratio)
        response = (
            f"Diabetic foot ulcer detected. "
            f"Wound coverage: {wound_ratio:.1%}. "
            f"Severity: {sev}/10. "
            f"{'Immediate' if sev >= 7 else 'Urgent'} clinical evaluation recommended."
        )
    else:
        response = (
            "No wound detected. "
            "Foot appears healthy with intact skin. "
            "Severity: 0/10. Continue routine monitoring."
        )
    return {
        "image_path": img_path,
        "response": response,
        "is_ulcer": is_ulcer,
    }


samples = []

# Dataset 1: 폴더 기반 이진 분류
for folder in DFU_DIR.rglob("*"):
    if not folder.is_dir():
        continue
    name = folder.name.lower()
    if any(k in name for k in ULCER_KEYWORDS):
        label = True
    elif any(k in name for k in HEALTHY_KEYWORDS):
        label = False
    else:
        continue
    for img in folder.glob("*.jpg"):
        samples.append(make_sample(str(img), label))
    for img in folder.glob("*.png"):
        samples.append(make_sample(str(img), label))

print(f"DFU binary: {len(samples)}개")

# Dataset 2: 세그멘테이션 마스크
seg_image_dir = next((d for d in SEG_DIR.rglob("images") if d.is_dir()), None)
seg_mask_dir  = next((d for d in SEG_DIR.rglob("masks")  if d.is_dir()), None)

seg_count = 0
if seg_image_dir and seg_mask_dir:
    for img_path in sorted(seg_image_dir.glob("*.jpg")):
        mask_path = seg_mask_dir / f"{img_path.stem}.png"
        if mask_path.exists():
            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            _, binary = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
            ratio = np.count_nonzero(binary) / binary.size
        else:
            ratio = 0.0
        samples.append(make_sample(str(img_path), is_ulcer=True, wound_ratio=ratio))
        seg_count += 1

print(f"Segmentation: {seg_count}개")
print(f"총 샘플: {len(samples)}개")

random.shuffle(samples)
split = int(len(samples) * 0.85)
train_samples = samples[:split]
val_samples   = samples[split:]
print(f"Train: {len(train_samples)} / Val: {len(val_samples)}")

DFU binary: 1835개
Segmentation: 0개
총 샘플: 1835개
Train: 1559 / Val: 276


In [8]:
# ── 5. HuggingFace Dataset 변환 ───────────────────────────────────────────────
# PIL Image를 content dict 안에 직접 넣으면 Dataset 직렬화 시 손상됨
# 해결: "file://절대경로" 문자열로 넣으면 process_vision_info가 직접 로드함

def to_hf_format(sample: dict) -> dict:
    abs_path = str(Path(sample["image_path"]).resolve())
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": f"file://{abs_path}"},
                {"type": "text",  "text": PROMPT},
            ],
        },
        {
            "role": "assistant",
            "content": [{"type": "text", "text": sample["response"]}],
        },
    ]
    return {"messages": messages}


train_dataset = Dataset.from_list([to_hf_format(s) for s in train_samples])
val_dataset   = Dataset.from_list([to_hf_format(s) for s in val_samples])

print(f"HF Dataset 준비 완료: train={len(train_dataset)}, val={len(val_dataset)}")
print("샘플 경로 확인:", train_dataset[0]["messages"][0]["content"][0]["image"][:60])

HF Dataset 준비 완료: train=1559, val=276
샘플 경로 확인: file:///kaggle/input/datasets/laithjj/diabetic-foot-ulcer-df


In [9]:
# ── 6. Gemma 4 E4B 모델 로드 + LoRA 설정 ─────────────────────────────────────
# 한 셀에서 실행 — 재실행해도 모델을 새로 불러오므로 LoRA 중복 에러 없음
model, processor = FastVisionModel.from_pretrained(
    "unsloth/gemma-4-E4B-it",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=42,
)
model.print_trainable_parameters()

==((====))==  Unsloth 2026.4.8: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

trainable params: 41,222,144 || all params: 8,037,378,592 || trainable%: 0.5129


In [10]:
# ── 8. Trainer 설정 ───────────────────────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    tokenizer=processor,
    data_collator=UnslothVisionDataCollator(model, processor),
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        fp16=True,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=100,
        output_dir="/kaggle/working/woundwatch-checkpoints",
        report_to="none",
        remove_unused_columns=False,
        dataset_kwargs={"skip_prepare_dataset": True},
    ),
)
print("Trainer 준비 완료")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Model does not have a default image size - using 512
Trainer 준비 완료


In [11]:
# ── 9. 파인튜닝 실행 ──────────────────────────────────────────────────────────
trainer_stats = trainer.train()
print(f"\n훈련 완료")
print(f"총 스텝: {trainer_stats.global_step}")
print(f"최종 Loss: {trainer_stats.training_loss:.4f}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,559 | Num Epochs = 3 | Total steps = 585
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,222,144 of 8,037,378,592 (0.51% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Step,Training Loss,Validation Loss
50,0.120259,4.241201
100,0.008618,4.492458
150,0.001394,4.332178
200,0.001393,4.482303
250,0.001021,4.502497
300,0.000538,4.451146
350,0.000685,4.437665
400,0.000511,4.456907
450,0.000273,4.444535
500,0.000262,4.455148


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/woundwatch-checkpoints/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/woundwatch-checkpoints/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/woundwatch-checkpoints/checkpoint-300/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/woundwatch-checkpoints/checkpoint-400/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/woundwatch-checkpoints/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/woundwatch-checkpoints/checkpoint-585/tokenizer_config.json.



훈련 완료
총 스텝: 585
최종 Loss: 0.4327


In [14]:
# ── 8. 추론 테스트 ───────────────────────────────────────────────────────────
FastVisionModel.for_inference(model)

test_img_path = train_samples[0]["image_path"]
test_image = Image.open(test_img_path).convert("RGB")

# Gemma4Processor는 content 안 이미지를 자동으로 추출함
# images= 를 따로 넘기면 중복 충돌 → 넣지 않음
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": test_image},
            {"type": "text",  "text": PROMPT},
        ],
    }
]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=False,   # temperature 제거 → greedy decoding
)
result = processor.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
)

print("=== 추론 결과 ===")
print(result)

=== 추론 결과 ===
This is an analysis of the provided image, which appears to show a severe wound on a lower leg, consistent with a diabetic foot ulcer.

**Disclaimer:** I am an AI and not a medical professional. This analysis is for informational purposes only and **does not substitute for professional medical diagnosis or treatment.** Immediate consultation with a qualified healthcare provider (such as a podiatrist, wound care specialist, or vascular surgeon) is essential.

---

### Wound Analysis

**(1) Wound Presence:**
*   **Yes.** There is a large, complex, and chronic ulceration present on the lower leg.

**(2) Wound Coverage Percentage:**
*   **Estimated Coverage:** Approximately 60% to 75% of the visible skin surface in the image is affected by the wound/inflammation.

**(3) Severity Score (0–10):**
*   **Score: 8/10**
    *   **Rationale:** This wound is severe


In [15]:
# ── 11. HuggingFace Hub 업로드 ────────────────────────────────────────────────
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")

# HF 사용자명을 본인 아이디로 변경하세요
HF_REPO = "5seoyoung/woundwatch-gemma4-e4b"

model.push_to_hub(HF_REPO, token=hf_token)
processor.push_to_hub(HF_REPO, token=hf_token)

print(f"\n업로드 완료: https://huggingface.co/{HF_REPO}")

README.md:   0%|          | 0.00/571 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/5seoyoung/woundwatch-gemma4-e4b


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmp0loegvfx/tokenizer_config.json.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


업로드 완료: https://huggingface.co/5seoyoung/woundwatch-gemma4-e4b
